<a href="https://colab.research.google.com/github/bonobobit/qff2026/blob/main/string_matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A quantum algorithm for string matching.
Implementazione dell'algoritmo descritto in
Niroula, P., Nam, Y. A quantum algorithm for string matching. npj Quantum Inf 7, 37 (2021). https://doi.org/10.1038/s41534-021-00369-3

In [2]:
%pip install qiskit pylatexenc numpy qiskit_aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 84.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 106.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 80.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [qiskit_aer]


In [3]:
from qiskit import QuantumRegister, QuantumCircuit
import numpy as np
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator

In [13]:
# 1. Inizializzazione dei registri

# Otteniamo stringa e pattern
T = "100100"
P = "0101"

# l'algoritmo funziona solo se |T| è potenza di 2
if not (len(T) & (len(T) - 1) == 0):
  print("Errore: |T| deve essere una potenza di 2")

# Ricaviamo lunghezza di T e P; n sono i qubit necessari per rappresentare N
N = len(T)
M = len(P)
n = int(np.ceil(np.log2(N)))
# servono N/2 ancilla qubits per il fan-out e M qubits per Grover
a = max(N//2, M)

# Creiamo i registri
reg_K = QuantumRegister(n, name="k") # registro indice
reg_T = QuantumRegister(N, name="t") # registro stringa
reg_P = QuantumRegister(M, name="p") # registro pattern
reg_A = QuantumRegister(a, name="a") # registro ancilla

# Creiamo il circuito

qc = QuantumCircuit(reg_K, reg_T, reg_P, reg_A)

# Popoliamo i registri usando identità e bit-flip come indicato nel paper
for i in range(N):
  if T[i] == "1":
    qc.x(reg_T[i])
  elif T[i] == "0":
    qc.id(reg_T[i])
  else:
    print("Errore nella codifica della stringa T")

for i in range(M):
  if P[i] == "1":
    qc.x(reg_P[i])
  elif P[i] == "0":
    qc.id(reg_P[i])
  else:
    print("Errore nella codifica del pattern")

# 2a. Usiamo la Quantum Fourier Transform per mettere in superposizione i k
qc.append(QFTGate(n), range(n))

# 2b. Implementiamo lo shift ciclico

# Usiamo degli ancilla qubit per copiare il valore k
# e usarlo in parallelo su diversi CNOT
def fanout():
  fo = QuantumCircuit(1+N//2) # controllo + ancilla qubits
  # primo passo: copiamo dal controllo al primo ancilla
  fo.cx(0, 1)
  # passi successivi
  copied = 1
  while copied < (N//2):
    for i in range(copied):
      if (i + copied) < (N//2):
        fo.cx(1+i, copied + 1+i)
    copied *= 2
  return fo.to_gate() # per invertirlo

# Shift ciclico
def cyclicShift():
  cs = QuantumCircuit(n+N+a)
  for z in range(n):
    k = 2**z
    cs.append(fanout(), list(range(n + N, n + N + (N // 2)))) # copia k in reg_A

    # i = strati paralleli, j = blocchi, x = elementi dei blocchi (prima metà)
    for i in range(int(np.log2(N)-np.log2(k)), 0, -1): # ordine inverso perché leftward
      anc_id = 0
      for j in range(N//((2**i)*k)):
        for x in range(j*k*(2**i), (j*(2**i)+1)*k):
          cs.cswap(n+N+anc_id, n+x, n+ x+(2**(i-1))*k)
          anc_id += 1

    cs.append(fanout().inverse(), list(range(n + N, n + N + (N // 2)))) # ripulisce reg_A per il giro dopo
  return cs.to_gate()

def computation():
  comp = QuantumCircuit(n+N+M+a)
  # list(range(n+N)) per K e T, list(range(n + N + M, n + N + M + a)) per A
  comp.append(cyclicShift(), list(range(n + N)) + list(range(n + N + M, n + N + M + a)))
  # Compariamo i primi M caratteri di T nei diversi cicli con P
  for i in range(M):
    comp.cx(reg_T[i], reg_P[i]) # il CNOT esegue uno XOR, il risultato va nel secondo parametro
  return comp.to_gate()

# P è tutto 0 se il pattern e la sottostringa sono identici,
# altrimenti ha tanti 1 quanti i bit differenti


# 5. Isoliamo i casi di match usando l'algoritmo di Grover
def andTree():
  at = QuantumCircuit((M*2)-1)
  i = 0
  for result in range(M-1): # result è l'indice di A che scorro, l'ultimo non serve (M-1)
    at.ccx(i, i+1, M+result) # M+result è la posizione nel registro combinato
    i += 2
  return at.to_gate()

def oracle(s):
  orc = QuantumCircuit(M*2)
  orc.x(range(M)) # inverte P
  orc.append(andTree(), range((M*2)-1))  # combina i qubit di P in un qubit a
  orc.cx(orc((M*2)-2), orc((M*2)-1)) #phase kickback tra penultimo e ultimo a
  # uncomputation
  orc.append(andTree().inverse(), range((M*2)-1))
  orc.x.inverse(range(M))

def groverDiffusion():
  # per la diffusione basta rifare H su reg_K per annullarlo,
  # ma K è rimasto entangled con T e P tramite il ciclo di rotazioni
  # dobbiamo quindi prima annullare questo entanglement


  print() # tmp
  # per la diffusione serve invertire i primi passaggi
  # quindi conviene unirli in un unico blocco invertibile

# prepariamo l'ultimo ancilla qubit a |-> per il phase kickback
qc.x(reg_A[M - 1])
qc.h(reg_A[M - 1])

def grover():
  for i in (np.log(N))
    qc.append(computation(), range(len(qc))) # shift + XOR
    qc.append(oracle(), )
    uncomputation() # XOR + shift
    diffusion()



#qc.measure_all()
qc.draw('mpl')

SyntaxError: expected ':' (177371031.py, line 91)